# ECDS measurement profile and review

Reviewed 16 September 2026 on commit c4c7f2ca. Only non-identifying aggregates are returned.
The full findings are in `../../docs/ecds-measurement-review.md`.

The facts were built at 14:24 BST. The landing tables refreshed around 14:59 BST,
and the shared attendance model was also rebuilt. Comparing those different
versions produces false reconciliation failures and inflated recent coverage
denominators. The recorded source comparison and coverage below use the landing
tables at **2026-09-16 13:24:20 UTC**, matching the fact build.

Snowflake retains these source versions for one day. After that window the saved
aggregate evidence remains inspectable, but the exact historical source queries
cannot be rerun. For a new review, use a completed source delivery and matching
fact/attendance builds, then select their matching source timestamp. Never treat
a changed denominator as a trend without checking refresh alignment.


## Execution

Use the existing `dbt-admin` Snow CLI connection and non-interactive authentication
configured outside this notebook. No credentials are stored here. `run_sql` accepts
only the reviewed aggregate queries below. No patient-level previews are needed.
The saved JSON outputs were obtained through Snow CLI during the review; notebook
cells have not been executed by a notebook kernel.


In [ ]:
import json
import subprocess
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / 'dbt_project.yml').exists():
    if ROOT == ROOT.parent:
        raise RuntimeError('Open this notebook within the dbt-analytics checkout')
    ROOT = ROOT.parent

def run_sql(sql):
    result = subprocess.run(
        ['snow', 'sql', '-c', 'dbt-admin', '--format', 'json', '-q', sql],
        check=True, capture_output=True, text=True, cwd=ROOT,
    )
    return json.loads(result.stdout)


## Same-snapshot source reconciliation and coverage

Checks every key and source payload field against the source at the build time.
Coverage means any submitted record among delivered attendances, not whether
care was provided. Provider comparisons use 2025 major emergency departments.


In [ ]:
snapshot_sql = 'select \'observation_snapshot_reconciliation\' as profile, count(*) as joined_rows,\n count_if(s.PRIMARYKEY_ID is null) as fact_only, count_if(f.visit_occurrence_id is null) as source_only,\n count_if(s.PRIMARYKEY_ID is not null and f.visit_occurrence_id is not null and not (equal_null(s."code",f.observation_code) and equal_null(s."value",f.observation_value) and equal_null(s."timestamp",f.observed_at) and equal_null(s."is_code_approved",f.is_code_approved) and equal_null(s.ROWNUMBER_ID,f.source_row_id) and equal_null(s."dmicImportLogId",f.dmic_import_log_id) and equal_null(s."ucum_unit_of_measurement",f.ucum_unit_code))) as altered_fields,\n count_if(f.visit_occurrence_id is not null and f.sk_patient_id is null) as missing_patient_key,\n count_if(f.visit_occurrence_id is not null and f.observed_at is null) as missing_timestamp\nfrom "Data_Store_SUS_Unified"."ECDS"."clinical.coded_observations" at (timestamp => \'2026-09-16 13:24:20 +00:00\'::timestamp_tz) s\nfull outer join DEV__REPORTING.ACUTE.FCT_SUS_UEC_OBSERVATION f on s.PRIMARYKEY_ID=f.visit_occurrence_id and s."CODED_OBSERVATIONS_ID"=f.source_sequence;\n\nselect \'assessment_snapshot_reconciliation\' as profile, count(*) as joined_rows,\n count_if(s.PRIMARYKEY_ID is null) as fact_only, count_if(f.visit_occurrence_id is null) as source_only,\n count_if(s.PRIMARYKEY_ID is not null and f.visit_occurrence_id is not null and not (equal_null(s."tool_type.code",f.assessment_tool_code) and equal_null(s."person_score",f.person_score) and equal_null(s."validation_timestamp",f.validated_at) and equal_null(s."tool_type.is_code_approved",f.is_code_approved) and equal_null(s.ROWNUMBER_ID,f.source_row_id) and equal_null(s."dmicImportLogId",f.dmic_import_log_id))) as altered_fields,\n count_if(f.visit_occurrence_id is not null and f.sk_patient_id is null) as missing_patient_key,\n count_if(f.visit_occurrence_id is not null and f.validated_at is null) as missing_timestamp\nfrom "Data_Store_SUS_Unified"."ECDS"."clinical.coded_scored_assessments" at (timestamp => \'2026-09-16 13:24:20 +00:00\'::timestamp_tz) s\nfull outer join DEV__REPORTING.ACUTE.FCT_SUS_UEC_SCORED_ASSESSMENT f on s.PRIMARYKEY_ID=f.visit_occurrence_id and s."CODED_SCORED_ASSESSMENTS_ID"=f.source_sequence;\n\nwith e as (\nselect PRIMARYKEY_ID as visit_occurrence_id, "attendance.arrival.date"::date as start_date,\n "attendance.location.department_type" as department_type,\n "attendance.location.hes_provider_3" as provider,\n "patient.age_at_arrival" as age_at_event\nfrom "Data_Store_SUS_Unified"."ECDS"."emergency_care" at (timestamp => \'2026-09-16 13:24:20 +00:00\'::timestamp_tz)\n), o as (select visit_occurrence_id from DEV__REPORTING.ACUTE.FCT_SUS_UEC_OBSERVATION group by 1),\na as (select visit_occurrence_id from DEV__REPORTING.ACUTE.FCT_SUS_UEC_SCORED_ASSESSMENT group by 1)\nselect \'snapshot_annual_coverage\' as profile,year(e.start_date) as attendance_year,\n count(*) as attendances,count(distinct e.visit_occurrence_id) as attendance_keys,\n count_if(o.visit_occurrence_id is not null) as with_observation,count_if(a.visit_occurrence_id is not null) as with_assessment,\n count_if(e.department_type=\'01\') as type1_attendances,\n count_if(e.department_type=\'01\' and o.visit_occurrence_id is not null) as type1_with_observation,\n count_if(e.department_type=\'01\' and a.visit_occurrence_id is not null) as type1_with_assessment\nfrom e left join o using(visit_occurrence_id) left join a using(visit_occurrence_id) group by 2 order by 2;\n\nwith e as (\nselect PRIMARYKEY_ID as visit_occurrence_id, "attendance.arrival.date"::date as start_date,\n "attendance.location.department_type" as department_type,\n "attendance.location.hes_provider_3" as provider,\n "patient.age_at_arrival" as age_at_event\nfrom "Data_Store_SUS_Unified"."ECDS"."emergency_care" at (timestamp => \'2026-09-16 13:24:20 +00:00\'::timestamp_tz)\n), o as (select visit_occurrence_id from DEV__REPORTING.ACUTE.FCT_SUS_UEC_OBSERVATION group by 1),\na as (select visit_occurrence_id from DEV__REPORTING.ACUTE.FCT_SUS_UEC_SCORED_ASSESSMENT group by 1)\nselect \'snapshot_provider_2025\' as profile,provider,count(*) as attendances,\n count_if(o.visit_occurrence_id is not null) as with_observation,count_if(a.visit_occurrence_id is not null) as with_assessment\nfrom e left join o using(visit_occurrence_id) left join a using(visit_occurrence_id)\nwhere start_date>=\'2025-01-01\' and start_date<\'2026-01-01\' and department_type=\'01\'\ngroup by 2 having count(*)>=10000 order by attendances desc;\n\nwith e as (\nselect PRIMARYKEY_ID as visit_occurrence_id, "attendance.arrival.date"::date as start_date,\n "attendance.location.department_type" as department_type,\n "attendance.location.hes_provider_3" as provider,\n "patient.age_at_arrival" as age_at_event\nfrom "Data_Store_SUS_Unified"."ECDS"."emergency_care" at (timestamp => \'2026-09-16 13:24:20 +00:00\'::timestamp_tz)\n), o as (select visit_occurrence_id from DEV__REPORTING.ACUTE.FCT_SUS_UEC_OBSERVATION group by 1),\na as (select visit_occurrence_id from DEV__REPORTING.ACUTE.FCT_SUS_UEC_SCORED_ASSESSMENT group by 1)\nselect \'snapshot_monthly_2026\' as profile,date_trunc(\'month\',start_date)::date as attendance_month,count(*) as attendances,\n count_if(o.visit_occurrence_id is not null) as with_observation,count_if(a.visit_occurrence_id is not null) as with_assessment\nfrom e left join o using(visit_occurrence_id) left join a using(visit_occurrence_id)\nwhere start_date>=\'2026-01-01\' group by 2 order by 2;\n'
snapshot_results = run_sql(snapshot_sql)
snapshot_results


## Fact profiles

The dbt analysis contains source/fact freshness diagnostics, temporal and provider
coverage, terminology matching, numeric parse status, repeated signatures and
unit resolution. Its live reconciliation and attendance coverage are deliberately
not saved below because the inputs had different refresh times. Use the
same-snapshot results above for those conclusions. Quantiles across mixed units
are diagnostics, not clinically comparable measurements.


In [ ]:
# Compile with the existing tracked DEV target before running this cell:
# dbt compile --target dev --profiles-dir . -s ecds_measurement_profile
sql = (ROOT / 'target/compiled/wnl_analytics/analyses/acute/ecds_measurement_profile.sql').read_text()
results = run_sql(sql)
# Return only the fact-based profiles; review live checks separately for refresh skew.
profiles = [group for group in results if group and group[0]['PROFILE'] in {
    'observation_terminology', 'observation_code_profile', 'observation_repeated_signatures',
    'assessment_terminology', 'assessment_code_profile', 'assessment_repeated_signatures',
    'unit_terminology', 'categorical_responses', 'unit_by_observation'
}]
profiles


## Unit and assessment family diagnostics

The measurements below are grouped by reported unit. Do not convert values or
repair codes from these distributions. In particular, the unlabelled temperature
distribution suggests mixed unit conventions but does not identify the unit of
an individual record.


In [ ]:
targeted_sql = """
select 'assessment_family' as profile, coalesce(measurement_category,code_description_source) as family,count(*) as records,count(distinct visit_occurrence_id) as attendances from DEV__REPORTING.ACUTE.FCT_SUS_UEC_SCORED_ASSESSMENT group by 2 order by records desc;
select 'measurement_units' as profile, observation_code,ucum_unit_code,unit_match_status,resolved_unit_symbol,count(*) as records,round(approx_percentile(observation_value_numeric,0.5),2) as median_value,round(approx_percentile(observation_value_numeric,0.99),2) as p99_value from DEV__REPORTING.ACUTE.FCT_SUS_UEC_OBSERVATION where observation_code in ('276885007','86290005','72313002') group by 2,3,4,5 having count(*)>=10000 order by 2,records desc;
select 'unmatched_provider' as profile,organisation_id,count(*) as records,count_if(is_code_approved) as source_approved from DEV__REPORTING.ACUTE.FCT_SUS_UEC_SCORED_ASSESSMENT where code_description_source='unmatched' group by 2 having count(*)>=1000 order by records desc;
"""
run_sql(targeted_sql)


## Score and time diagnostics

NEWS2 component checks use the [RCP scoring chart](https://rcp.ac.uk/media/alxev00t/news2-chart-1_the-news-scoring-system_0_0.pdf).
They flag scores outside the published discrete points, not clinical severity.
Dates before arrival or over a week later warrant review; they are not automatically
invalid because delayed validation and linked care can be legitimate.


In [ ]:
semantic_sql = '-- Published NEWS2 component points are whole numbers 0 to 3; oxygen is 0 or 2 and consciousness is 0 or 3.\nselect \'news2_component_validity\' as profile, assessment_tool_code, assessment_description,\n count(*) as records,\n count_if(person_score_numeric is not null) as numeric_records,\n count_if(person_score_numeric is not null and (\n   person_score_numeric != trunc(person_score_numeric)\n   or person_score_numeric < 0 or person_score_numeric > 3\n   or (assessment_tool_code=\'1104331000000105\' and person_score_numeric not in (0,2))\n   or (assessment_tool_code=\'1104361000000100\' and person_score_numeric not in (0,3)))) as outside_component_values\nfrom DEV__REPORTING.ACUTE.FCT_SUS_UEC_SCORED_ASSESSMENT\nwhere ecds_group1=\'NEWS2\' and assessment_tool_code!=\'1104051000000101\'\ngroup by 2,3 order by outside_component_values desc;\nselect \'event_time_quality\' as profile, \'observation\' as record_type,count(*) as records,\n count_if(observed_at::date<attendance_date) as before_arrival_date,\n count_if(observed_at::date>dateadd(day,7,attendance_date)) as over_week_after_arrival,\n count_if(year(observed_at)<2018) as before_ecds_history,\n count_if(sk_patient_id is null) as missing_patient_key\nfrom DEV__REPORTING.ACUTE.FCT_SUS_UEC_OBSERVATION\nunion all select \'event_time_quality\',\'assessment\',count(*),\n count_if(validated_at::date<attendance_date),\n count_if(validated_at::date>dateadd(day,7,attendance_date)),\n count_if(year(validated_at)<2018),count_if(sk_patient_id is null)\nfrom DEV__REPORTING.ACUTE.FCT_SUS_UEC_SCORED_ASSESSMENT;\nselect \'source_key_churn\' as profile,count(*) as fact_observations,\n count_if(e.PRIMARYKEY_ID is null) as missing_current_attendance\nfrom DEV__REPORTING.ACUTE.FCT_SUS_UEC_OBSERVATION f\nleft join "Data_Store_SUS_Unified"."ECDS"."emergency_care" e on f.visit_occurrence_id=e.PRIMARYKEY_ID;\n'
run_sql(semantic_sql)


## External terminology check

On 16 September 2026, the configured UK SNOMED FHIR server returned 404 for
`1104051000000100` and `1104331000000100`. A control lookup of
`1104051000000101` returned the NEWS2 total-score term, edition
`http://snomed.info/sct/83821000000107/version/20260826`.
This supports leaving the first two codes unresolved. Similarity to valid codes
does not authorise a correction. No patient data was sent to the server.

```powershell
pwsh ~/scripts/terminology-query.ps1 lookup 1104051000000100
pwsh ~/scripts/terminology-query.ps1 lookup 1104331000000100
pwsh ~/scripts/terminology-query.ps1 lookup 1104051000000101
```


## Recorded live refresh mismatch
These diagnostics compare the old facts with the newer source delivery. They do not establish transformation loss. The same-snapshot checks above resolve that question.


In [ ]:
live_sql = "-- Aggregate-only profile. No patient keys or row-level values leave Snowflake.\n-- Coverage denominators are delivered ECDS attendances, not all clinical care.\n-- Live comparisons require aligned source and fact refreshes. For this review, use\n-- the same-snapshot reconciliation and coverage in the companion notebook.\n-- Numeric quantiles across reported units are diagnostic, not standardised measures.\n\nselect 'observation_live_reconciliation' as profile,\n count(*) as joined_rows,\n count_if(s.visit_occurrence_id is null) as fact_only_rows,\n count_if(f.visit_occurrence_id is null) as source_only_rows,\n count_if(s.visit_occurrence_id is not null and f.visit_occurrence_id is not null and (not equal_null(s.observation_code, f.observation_code) or not equal_null(s.observation_value, f.observation_value) or not equal_null(s.observed_at, f.observed_at) or not equal_null(s.is_code_approved, f.is_code_approved) or not equal_null(s.source_row_id, f.source_row_id) or not equal_null(s.dmic_import_log_id, f.dmic_import_log_id) or not equal_null(s.ucum_unit_code, f.ucum_unit_code))) as altered_source_fields,\n count_if(f.visit_occurrence_id is not null and f.sk_patient_id is null) as patient_key_missing,\n count_if(f.visit_occurrence_id is not null and f.observed_at is null) as timestamp_missing,\n count_if(f.observed_at::date > current_date()) as future_timestamp_rows,\n count_if(f.observed_at::date < f.attendance_date) as before_arrival_date,\n count_if(f.observed_at::date > dateadd(day, 7, f.attendance_date)) as over_week_after_arrival,\n min(date_trunc('month',f.observed_at))::date as first_event_month,\n max(date_trunc('month',f.observed_at))::date as last_event_month\nfrom DEV__STAGING.SUS.stg_sus_ecds_clinical_coded_observations s\nfull outer join DEV__REPORTING.ACUTE.fct_sus_uec_observation f\n on s.visit_occurrence_id=f.visit_occurrence_id and s.source_sequence=f.source_sequence;\n\nselect 'assessment_live_reconciliation' as profile,\n count(*) as joined_rows,\n count_if(s.visit_occurrence_id is null) as fact_only_rows,\n count_if(f.visit_occurrence_id is null) as source_only_rows,\n count_if(s.visit_occurrence_id is not null and f.visit_occurrence_id is not null and (not equal_null(s.assessment_tool_code, f.assessment_tool_code) or not equal_null(s.person_score, f.person_score) or not equal_null(s.validated_at, f.validated_at) or not equal_null(s.is_code_approved, f.is_code_approved) or not equal_null(s.source_row_id, f.source_row_id) or not equal_null(s.dmic_import_log_id, f.dmic_import_log_id))) as altered_source_fields,\n count_if(f.visit_occurrence_id is not null and f.sk_patient_id is null) as patient_key_missing,\n count_if(f.visit_occurrence_id is not null and f.validated_at is null) as timestamp_missing,\n count_if(f.validated_at::date > current_date()) as future_timestamp_rows,\n count_if(f.validated_at::date < f.attendance_date) as before_arrival_date,\n count_if(f.validated_at::date > dateadd(day, 7, f.attendance_date)) as over_week_after_arrival,\n min(date_trunc('month',f.validated_at))::date as first_event_month,\n max(date_trunc('month',f.validated_at))::date as last_event_month\nfrom DEV__STAGING.SUS.stg_sus_ecds_clinical_coded_scored_assessments s\nfull outer join DEV__REPORTING.ACUTE.fct_sus_uec_scored_assessment f\n on s.visit_occurrence_id=f.visit_occurrence_id and s.source_sequence=f.source_sequence;"
run_sql(live_sql)
